# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates the process of loading, exploring, and analyzing the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) mass spectrometry protein dataset using the `mlcroissant` library. All dataset entities are referenced using their `@id` (per ML Commons Croissant best practices).

### Dataset Source
The dataset is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
This section reviews the available record sets, their fields, and all relevant Croissant `@id`s (identifiers).

The dataset may contain one or multiple record sets. We enumerate all, print their `@id`s, and their fields.

In [ ]:
# List all record sets with their @id
record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record set(s).\n")
for rs in record_sets:
    print(f"RecordSet name: {rs.name!r} | @id: {rs.id}")
    print("  Fields (by @id):")
    for field in rs.fields:
        print(f"    - {field.name} (id: {field.id}) | type: {field.data_type}")
    print()
# Save the first record_set @id for further use
if len(record_sets) > 0:
    first_recordset_id = record_sets[0].id
    # Collect field ids for this recordset
    recordset_field_ids = [f.id for f in record_sets[0].fields]

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames. We use the record set and field `@id`s (see above) for unambiguous referencing.

_Note_: Actual data records are accessible via the Croissant schema and linked files; some large record sets may require additional resources or may not fully download in sandboxed cloud environments.

In [ ]:
# Extract data from RecordSets into pandas DataFrames
# For demonstration, only load the first record set (if present)
import itertools
dataframes = {}

if len(record_sets) == 0:
    print("No record sets are defined in this dataset.")
else:
    record_set_ids = [rs.id for rs in record_sets]
    for rid in record_set_ids:
        # Use the mlcroissant API to iterate records
        print(f"Loading data for record set: {rid}")
        records = list(dataset.records(record_set=rid))
        if records:
            df = pd.DataFrame(records)
            dataframes[rid] = df
            print(f"  Columns: {list(df.columns)}\n  Records: {len(df)}\n")
        else:
            print(f"  No records found for record set {rid}.")
# Display head of the first recordset dataframe (if available)
if dataframes:
    example_id = list(dataframes)[0]
    print(f"Example preview of '{example_id}':")
    display(dataframes[example_id].head(3))

## 4. Exploratory Data Analysis (EDA)
Let's perform basic data processing/EDA on the first main record set. We'll select one numeric field (by `@id`), filter on its values, normalize the field, and group by a categorical field if present.

In [ ]:
# EDA: Filter, normalize, and group
import numpy as np

if not dataframes:
    print("No data to analyze.")
else:
    record_set_id = list(dataframes)[0]  # Use the first record set
    df = dataframes[record_set_id]

    # Find numeric fields by brute force (int/float columns)
    numeric_field_id = None
    numeric_fields = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_fields.append(col)
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Use first numeric field
        print(f"Selected numeric field: '{numeric_field_id}'\n")
        # Pick a threshold - e.g., 10th percentile
        threshold = np.nanpercentile(df[numeric_field_id], 10)
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with '{numeric_field_id}' > {threshold:.3f}:")
        display(filtered_df.head())

        # Add normalized column
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
        print(f"Normalized '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a likely categorical field (first non-numeric column)
        group_field_id = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame().reset_index()
            print(f"Grouped by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No categorical fields available for grouping.")
    else:
        print("No numeric fields found in the data.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and demonstrate a grouped bar chart if feasible. If the dataset is large, we only plot the first 1000 records.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_fields:
    sample = filtered_df.head(1000)
    plt.figure(figsize=(7,4))
    sns.histplot(sample[numeric_field_id], kde=True, bins=30)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.tight_layout()
    plt.show()
    # If group field is present, show group means
    if group_field_id:
        order = grouped_df[group_field_id] if grouped_df.shape[0] < 10 else None
        plt.figure(figsize=(9,4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, order=order)
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean '{numeric_field_id}' by '{group_field_id}'")
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("Insufficient data for plotting.")

## 6. Conclusion

In this notebook, we've loaded a mass spectrometry dataset described with the Croissant schema, reviewed its record sets and field structure, and performed initial exploratory analysis and visualization using convenient `mlcroissant` APIs. Further analysis may include deeper statistical summaries, domain-specific visualizations, or export to additional ML pipelines.

**Key takeaways:**
- All entities are referenced programmatically by their Croissant `@id` for reproducibility.
- The rich schema allows for robust data access and flexible exploration using the `mlcroissant` library.

_Next steps_: Consider integrating record provenance, detailed field descriptions, or linking direct provenance/data citations if required for your downstream application.